# 6.5940 · Long-context rank follow-up

This CPU replay compares the original and corrected rank baseline at B=96. It also recomputes the reuse-prediction AUC with a tie correction and a complete-window sensitivity check.

You need `rank-followup-colab.zip` (code and archived controls) and the **original 14 `longctx_v1` dev NPZs** in a single Drive folder. These are the long-context traces with `rid`, `ranked`, and `scores`. The earlier top-K-only Colab export is for a different split.

Start with `SMOKE = True` to compare one layer against the archived LRU and Bélády counts. Then switch it to `False` and rerun the configuration and replay cell. Results are written to Drive. A CPU runtime is sufficient.

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import io

from google.colab import drive, files

drive.mount('/content/drive')
uploaded = files.upload()
bundle = uploaded.get('rank-followup-colab.zip')
if bundle is None:
    raise ValueError('Upload rank-followup-colab.zip in this cell.')
with ZipFile(io.BytesIO(bundle)) as source:
    source.extractall('/content')
project = Path('/content/rank-followup')
assert (project / 'validation/rerun_longctx_dev.py').is_file()

In [ ]:
from datetime import datetime, timezone
import subprocess
import sys

# Set this to the Drive folder with the NPZ files directly inside it.
TRACES_DIR = ''
SMOKE = True

traces = Path(TRACES_DIR)
if not TRACES_DIR or not any(traces.glob('*.npz')):
    raise ValueError('Set TRACES_DIR to the Drive folder containing the original dev NPZs.')
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
mode = 'smoke' if SMOKE else 'full'
output = Path('/content/drive/MyDrive/rank-kv-followup') / f'{mode}-{stamp}'
cmd = [sys.executable, '-u', 'validation/rerun_longctx_dev.py', str(traces), '--out', str(output)]
if SMOKE:
    cmd.append('--smoke')
with subprocess.Popen(cmd, cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as proc:
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('Replay failed. See the trace or control mismatch above.')
print((output / 'summary.json').read_text())

A smoke run writes `rank_by_layer.csv` and `summary.json`. The full run also writes `rank_lodo.csv` and `auc.csv`. Compare the original and corrected LODO entries in `summary.json` before updating any poster result. Keep the complete-window AUC column separate from the tie-only comparison.